# Day 3 模块 3：从一棵树到一片森林

一棵树不够稳，就请很多棵树一起判断。


In [1]:
from pathlib import Path
import pandas as pd

candidate_paths = [
    Path('day3_cafe_sales.csv'),
    Path('day3') / 'day3_cafe_sales.csv',
    Path('教学课程') / 'day3' / 'day3_cafe_sales.csv',
]
for path in candidate_paths:
    if path.exists():
        csv_path = path
        break
else:
    raise FileNotFoundError('未找到 day3_cafe_sales.csv')

df = pd.read_csv(csv_path)
print(csv_path.resolve())
print('shape:', df.shape)
df.head()


D:\obi\熵学院\授课\机器学习-6小时\教学课程\day3\day3_cafe_sales.csv
shape: (200, 12)


,day,weather,weather_label,temp,is_weekend,promotion,quality,price,location,competitors,holiday,sales
0,1,1,多云,33.5,0,1,8.27,24.5,5.78,2,0,3249
1,2,2,阴,6.7,0,1,9.57,22.0,9.96,0,0,4700
2,3,0,晴,23.4,0,0,7.11,27.9,7.28,5,0,2071
3,4,2,阴,6.4,0,0,6.65,23.0,9.74,3,0,2370
4,5,0,晴,11.9,0,0,8.60,31.8,9.17,2,0,2847


## 1. 为什么要很多棵树

上一模块：一棵树会在数据上**反复找最好的切分**，长出一串 if。

问题是：单个同学做判断，可能偏差大。

如果：

- 很多同学各自看到略有不同的练习题  
- 每人在自己的题上长出一棵树（各自做同一套「切刀学习」）  
- 最后取平均（回归）或投票（分类）  

整体往往会更稳。

随机森林就是这个思路。


## 2. Bagging：先记住白话

**Bagging** 可以先记成：

1. 从原数据里反复抽样，得到很多“略有不同”的子集  
2. 每个子集训练一棵树（每棵树都在自己的子集上搜索切分）  
3. 预测时把所有树的结果合起来  

比喻：

> 三个臭皮匠，顶个诸葛亮。

所以：

> 森林会学，是因为里面的每一棵树都会学；  
> 森林更稳，是因为很多棵一起平均。


## 3. 特征也随机一点（每个节点都抽）

随机森林除了**样本随机**（Bagging），还有**特征随机**。

### 怎么抽？

设全部特征有 \(M\) 个（本课大约 9 个：价格、促销、周末……）。  
每次要切一刀时，只随机抽出 \(m\) 个特征当候选（\(m \le M\)），**只在这 \(m\) 个里**用上一模块的标准选最好的刀（让 \(E_后\) 尽量小 / Gain 尽量大）。

重要：

- **不是**整棵树开头抽死一组特征，整棵树只能用它们  
- **而是每个要切分的节点**都重新从 \(M\) 里抽一次 \(m\)  
- 根节点抽一拨；左孩子、右孩子再切时，各自再抽，可以完全不同

```text
根节点：从 M 抽 m → 选一刀
  ├─ 左：再从 M 抽 m → 再选一刀
  └─ 右：再从 M 抽 m → 再选一刀
```

代码里对应参数常叫 `max_features`（控制 \(m\)）。今天先知道含义，不必纠结最优取值。

### 为什么 \(m\) 往往小于 \(M\)？

**主因不是省计算**，而是：

> 让每棵树别总盯着同一个“最强特征”，结构更不一样，平均起来才更稳。

如果每次都看全部特征，很多棵树的第一刀可能都切同一个特征 → 意见太像 → “三个臭皮匠”变弱。  
\(m\) 小一点，有时最强特征不在候选里，被迫用别的特征切 → 树更花。

省一点搜索时间是**顺带**的；`n_estimators=100` 其实还更费算。

今天记住三句即可：

1. 样本随机（每棵树一份略不同的数据）  
2. 特征随机（**每个节点**抽 \(m\) 个再选刀）  
3. 最后很多棵树的预测取平均


## 4. 随机森林今天要你会的三件事

1. 创建 `RandomForestRegressor`
2. `fit` 训练
3. `predict` 预测，并查看特征重要性

下一模块开始真正训练。


## 5. 小对照

| | 决策树 | 随机森林 |
| --- | --- | --- |
| 数量 | 1 棵 | 很多棵 |
| 稳定性 | 容易飘 | 通常更稳 |
| 可解释 | 更容易画出来 | 更靠特征重要性 |
| 今天角色 | 先修课 | 主模型 |
